In [ ]:
"""
Author: Michael Yao
Contributors: Ashish Puri

This script makes six-panel plots for an arbitrary input income stream
    (see Figure 12 in Guvenen, Karahan, Ozkan, Song (2021), *Econometrica*)
    (This is my best attempt at replicating their figure, but in some plots I'm off by 0.1-0.2.)

You should focus on the panels for standard deviation, skew, and kurtosis. 
    They will tell you if your model has good dynamic properties.

Don't worry if your bottom two panels (employment CDF, cross-section variance) don't match the data.
    - for employment: Guvenen assumes people can work 36 years total (ages 25-60)
        - if you assume a different total number of years, his CDF will no longer make sense
    - for cross-sectional variance: Guvenen calculates this by averaging across all the cohorts he has
        - you will probably plot a single cohort instead.

Code dependencies: 
- a01_parameters

Data dependencies: 
- data/intermediate/guvenenmoments
    - SdSkewKurt_L5.txt
    - meanLTinc_level.txt
    - EmpCDF.txt
    - var_lny.txt
    - GKOS_2016_moments_women.xlsx

"""

"""IMPORTS"""

# from a01_parameters import *

import statistics
from scipy import stats

"""
LOAD GUVENEN'S DATA MOMENTS
"""

"""MEN"""

# MOMENTS (PANELS A-C)

# df_guv_three is a dataframe containing the data for standard deviation, skew, kurtosis of five-year earnings (arc-percent) changes
# these are the same data moments Guvenen plots in Figure 12
# see documentation in /data for more info on the data used

# read the data from Guvenen's replication package
df_guv_three = pd.read_table(fp_data / 'intermediate' / 'guvenenmoments' / 'SdSkewKurt_L5.txt', header=None)
# name each column the corresponding statistic
df_guv_three = df_guv_three.rename(columns={0:"std", 1:"skew", 2:"kurt"})
# each column stores moments for young, middle-aged, and older workers. average the moments across the age groups
df_guv_three = 1/3*(df_guv_three.iloc[0:13].reset_index()+ df_guv_three.iloc[13:26].reset_index()+ df_guv_three.iloc[26:39].reset_index())

# EARNINGS GROWTH (PANEL D)

# read data from rep package containing mean earnings in (thousands of) dollars
guv_growth = pd.read_table(fp_data / 'intermediate' / 'guvenenmoments' / 'meanLTinc_level.txt', header=None).to_numpy()
# each column is an age (ages 25, 30, ..., 55, 60). second-to-last column is age 55, first column is age 25
# compute log earnings, then compute growth
guv_growth = np.log(guv_growth[:, -2]) - np.log(guv_growth[:, 0]) 

# EMPLOYMENT CDF (PANEL E)

emp_cdf = np.array(pd.read_table(fp_data / 'intermediate' / 'guvenenmoments' / 'EmpCDF.txt', header=None)[0])/100

# VARIANCE OF WITHIN-COHORT LOG EARNINGS (PANEL F)

var_lny = np.array(pd.read_table(fp_data / 'intermediate' / 'guvenenmoments' / 'var_lny.txt', header=None)[0])

"""WOMEN"""

# MOMENTS (PANELS A-C)

# df_guvf_three is the female equivalent of df_guv_three
# read the excel workbook with the data. load the sheet containing moments of five-year arc-percent changes
df_guvf_three = pd.read_excel(fp_data / 'intermediate' / 'guvenenmoments' / 'GKOS_2016_moments_women.xlsx', sheet_name = 'L5_arc_age_re')
# these moments are split up by recent earnings percentile and age group. we'll average out the age groups.
df_guvf_three = df_guvf_three.groupby('RE pctile').mean().reset_index()
# rename the columns to each moment
df_guvf_three = df_guvf_three.rename(columns = {'RE pctile' : 'RE', 'Standard deviation' : 'std', 'Skewness' : 'skew', 'Kurtosis' : 'kurt'}).drop('Age group', axis = 1)

# EARNINGS GROWTH (PANEL D)

# read the correct sheet
guvf_growth = pd.read_excel(fp_data / 'intermediate' / 'guvenenmoments' / 'GKOS_2016_moments_women.xlsx', sheet_name = 'incgrowth')
# rename columns
guvf_growth = guvf_growth.rename(columns = {'LE pctile' : 'LE', 'Growth 25-55' : 'LE_growth'})

# NO DATA FOR EMPLOYMENT CDF OR WITHIN-COHORT VARIANCE OF LOG EARNINGS

"""
SIX-PANEL PLOT
"""

def plot_six_moments(results, sex = 0, y_min = 1.5):
    """"this function calculates moments of income changes and makes a six-panel plot from a set of earnings streams you input.
    
    arguments:
    - results: a NumPy array of earnings histories, where rows are workers and columns are years
    - sex: gender of workers. 0 is men, 1 is women
    - y_min: a minimum income threshold. we drop all earnings observations below this threshold/earnings below this threshold count as non-employed 
        - if you're working directly with output from guv_model, I use y_min = 1.5
        - if you're working with dollar-valued output, use y_min = 2000

    example calls:
        plot_six_moments(guv_model_results, sex = 0, y_min = 1.5)
        plot_six_moments(cms_model_results, sex = 1, y_min = 2000)

    this function does not return anything. 
    it displays a six-panel plot.
    """

    """ 
    SET-UP
    """

    # count the number of workers
    n_workers = len(results)
    # to avoid changing the original input earnings streams, we'll work with a copy of the data
    sim_earnings = np.array(results)

    # make a vector of minimum thresholds, where each entry corresponds to an age.
    y_min_mat = y_min * np.ones(len(results[0])) 

    # count number of years in each earning stream
    num_years = len(results[0])
    
    # Guvenen's employment CDF data assumes you work a maximum of 36 years. 
    # We'll truncate his data. For example, if the input income streams are only for 24 years, we'll take the first 24 years of data in Guvenen's employment CDF
    emp_cdf_truncate = emp_cdf[:(num_years+1)]

    
    """
    MOMENTS:

    Figure 12a/b/c: moments of income 5year arc percent change vs recent earnings percentile
    """

    """age dummies: mean income for each age"""

    agedum = np.ones(num_years) 

    # loop over each age
    for i in range(num_years): #range(36):
        # age dummy for that age is the mean income, among everyone who earns above the minimum threshold
        #   more precisely, it's exp(mean(log income))
        agedum[i] = np.mean(np.log(sim_earnings[sim_earnings[:,i] >= y_min, i]))

    agedum = np.exp(agedum)

    # avgagedum will contain the average age dummy over the last five years, for each age
    avgagedum = np.ones(num_years-8) #np.ones(28)


    """sample selection"""

    longdata = pd.DataFrame(np.zeros(((num_years-8)*n_workers, 4)), columns=['RE', 'AP_change', 'h', 'RE_bin'])  
                #pd.DataFrame(np.zeros((28*n_workers, 4)), columns=['RE', 'AP_change', 'h', 'RE_bin']) 
    #shape data into long format.
    #first column for recent earnings, second for arc-percent change, third for age, fourth for recent earnings bin

    # loop over each age, omitting the first two years 
    for i in range(3, num_years-5): #range(3, 31): 

        # if we can look at earnings from the past five years, lag = 5
        # if the person is too young to have worked for five years, lag will be smaller
        lag = min(i, 5) #to calculate recent earnings for i-3, 4
        
        #select sample using the wide data
        
        #select only those who are above Y_min for at least two years

        #first create a mask
        mask_12abc = sim_earnings >= y_min_mat

        #we want people who are above the threshold in (t OR t + 5) AND (t - 1) AND (two years between t-5 and t-2)

        #sum across each row to count the years
        two_more_years = np.sum(mask_12abc[:, (i-lag, i-2)], axis=1) >= 2

        #must also be eligible in t-1, AND t OR t+5
        eligible = (((sim_earnings[:, i-1] >= y_min_mat[i-1]) & two_more_years) & 
                    ((sim_earnings[:, i] >= y_min_mat[i]) | (sim_earnings[:, i+5] >= y_min_mat[i+5])))
        
        sample_12abc = sim_earnings.copy() #make the slice a copy instead of a view
        sample_12abc[~eligible, :] = np.nan #mark ineligible rows NaN
        

        """average age dummies and recent earnings"""
        
        #calculate average age dummies for the five year period
        avgagedum[i-3] = np.mean(agedum[range(i-lag, i)])
        
        #calculate recent earnings using wide data, propagating NaN
        recent_earnings_mat = np.maximum(y_min_mat[range(i-lag, i)], sample_12abc[:, range(i-lag, i)])
        recent_earnings = np.mean(recent_earnings_mat, axis = 1)/avgagedum[i-3]

        """arc percent change"""
        
        #calculate arc percent change using wide data, propagating NaN
        AP_change = (2 * (sample_12abc[:, i+5]/agedum[i+5] - sample_12abc[:, i]/agedum[i]) / 
            (sample_12abc[:, i+5]/agedum[i+5] + sample_12abc[:, i]/agedum[i]))
            
        #store key numbers (RE, AP change, age) in long data    
        lb = (i-3)*n_workers #lower index of cohort in long data
        ub = (i-2)*n_workers #upper index
        longdata.iloc[lb:ub, 0] = recent_earnings
        longdata.iloc[lb:ub, 1] = AP_change
        longdata.iloc[lb:ub, 2] = i + 25 #age at time t

    """six age bins (between ages 27, 30, 35, 40, 45, 50, 55)"""

    #now define the age bins
    agebin = [int(s) for s in np.linspace(26,int(max(longdata['h'])),7)]
        #np.array([27, 30, 35, 40, 45, 50, 55])
    #define percentile bounds for RE bins
    RE_bins = np.array([0, 0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1])

    #define arrays for the three moments binned into six age bins
    std6mat = np.zeros((13, 6)); skew6mat = np.zeros((13, 6)); kurt6mat = np.zeros((13, 6))
    #arrays for the three age groups (of two bins each)
    std3mat = np.zeros((13, 3)); skew3mat = np.zeros((13, 3)); kurt3mat = np.zeros((13, 3))


    """calculate stdev, skew, kurtosis, conditional on recent earnings and age bin"""

    #loop through the age bins
    for i in range(len(agebin)-1):
        age_bin_data = longdata.query('h <= @agebin[@i+1] and h > @agebin[@i]').copy()
        
        #calculate percentiles
        RE_quantiles = np.nanquantile(age_bin_data['RE'], RE_bins)

        #loop through RE bins                           
        for j in range(len(RE_quantiles)-1):
            #get the locations of everyone in that bin
            RE_mask = np.array((age_bin_data['RE'] <= RE_quantiles[j+1]) & (age_bin_data['RE'] > RE_quantiles[j]))
            age_bin_data.iloc[RE_mask, 3] = j #record the correct bin in longdata
            #calculate moments, omitting NaN (which are ineligible and dropped from the rolling panel)
            std6mat[j, i] = np.nanstd(age_bin_data.iloc[RE_mask, 1])
            skew6mat[j, i] = stats.moment(age_bin_data.iloc[RE_mask, 1], 3, nan_policy='omit')
            kurt6mat[j, i] = stats.moment(age_bin_data.iloc[RE_mask, 1], 4, nan_policy='omit')
            
    #standardize third and fourth central moments by a power of the standard deviation
    skew6mat = skew6mat / (std6mat ** 3)
    kurt6mat = kurt6mat / (std6mat ** 4)

    """average six bins into three groups, then into a single figure"""

    #average the six bins into three age groups
    for i in range(3):
        std3mat[: ,i] = 0.5*(std6mat[:, 2*i] + std6mat[:, 2*i+1])
        skew3mat[:, i] = 0.5*(skew6mat[:, 2*i] + skew6mat[:, 2*i+1])
        kurt3mat[:, i] = 0.5*(kurt6mat[:, 2*i] + kurt6mat[:, 2*i+1])

    #now average across the three age groups
    growth_std = np.mean(std3mat, axis = 1)
    skew = np.mean(skew3mat, axis = 1)
    kurtosis = np.mean(kurt3mat, axis = 1)
    

    """12d lifecycle earnings growth"""

    """select sample by excluding individuals with frequently low earnings
    calculate lifetime earnings for each individual
    calculate quantiles for lifetime earnings
    mask by percentile and calculate mean earnings at 25 and at last
    find log difference"""

    #select only those who are above Y_min for at least 15 years
    #first create a mask
    mask_12d = sim_earnings >= y_min_mat
    #now sum across each row to count the years
    eligible_12d = np.sum(mask_12d, axis=1) >= 15
    sample_12d = sim_earnings[eligible_12d, :]

    #calculate lifetime earnings for each individual
    life_earnings = np.mean(sample_12d, axis = 1)
    #calculate percentiles
    LE_quantiles = statistics.quantiles(life_earnings, n=100)
    LE_bins = np.array([0, 4, 9, 19, 29, 39, 49, 59, 69, 79, 89, 94, 96, 98]) #14 cut points for 15 groups
    LE_quantiles = np.array(LE_quantiles)[LE_bins]

    #mean earnings at 25 by LE %ile, mean earnings at last by LE %ile
    earnings_25 = np.zeros(15); earnings_last = np.zeros(15)

    #mask by percentile and calculate mean earnings at 25 and last
    #first handle the first percentile
    LE_mask = life_earnings <= LE_quantiles[0]
    earnings_25[0] = np.mean(sample_12d[LE_mask, 0])
    earnings_last[0] = np.mean(sample_12d[LE_mask, num_years-1])

    for j in range(1, len(LE_quantiles)):
        LE_mask = (life_earnings <= LE_quantiles[j]) & (life_earnings >= LE_quantiles[j-1])
        earnings_25[j] = np.mean(sample_12d[LE_mask, 0])
        earnings_last[j] = np.mean(sample_12d[LE_mask, num_years-1])
    #now deal with the 100th percentile
    LE_mask = life_earnings > LE_quantiles[-1]
    earnings_25[-1] = np.mean(sample_12d[LE_mask, 0])
    earnings_last[-1] = np.mean(sample_12d[LE_mask, num_years-1])

    log_earnings_25 = np.log(earnings_25); log_earnings_last = np.log(earnings_last)

    log_growth = log_earnings_last - log_earnings_25

    """12e employment CDF"""
    employment = sim_earnings >= y_min
    years_employed = np.sum(employment, axis = 1)
    years_employed = np.sort(years_employed)
    cum_prob = np.linspace(0, 1, len(years_employed))

    """12f, within cohort variances of log earnings"""
    log_earnings = sim_earnings.copy() #initialize this matrix of log earnings. COPY of sim_earnings
    log_earnings[sim_earnings < y_min] = np.nan #make earnings under min threshold NaN
    log_earnings = np.log(log_earnings) #np.log(np.nan) == nan
    #take the log of earnings, ignoring NaN

    cohort_variances = np.nanvar(log_earnings, axis=0) #take variances of each column/cohort
    #ignore the NaN


    """plot results"""
    plt.style.use('ggplot')

    fig, ax = plt.subplots(3, 2)
    fig.set_size_inches(10, 18)

    RE_x_axis = [1, 6, 15.5, 25.5, 35.5, 45.5, 55.5, 65.5, 75.5, 85.5, 93, 97.5, 100]

    ax[0, 0].plot(RE_x_axis, growth_std, color='blue')
    ax[0, 0].set_title('Standard Deviation')
    ax[0, 0].set_xlabel('Percentile of Recent Earnings Distribution')
    ax[0, 0].set_ylabel('Std Dev of 5-year earnings arc-percent change')
    #ax[0, 0].set_ylim(0.3, 1.4)

    ax[0, 1].plot(RE_x_axis, skew, color='blue', label = "Model")
    ax[0, 1].set_title('Skew')
    ax[0, 1].set_xlabel('Percentile of Recent Earnings Distribution')
    ax[0, 1].set_ylabel('Skew of 5-year earnings arc-percent change')
    #ax[0, 1].set_ylim(-1, 0.2)

    ax[1, 0].plot(RE_x_axis, kurtosis, color='blue')
    ax[1, 0].set_title('Kurtosis')
    ax[1, 0].set_xlabel('Percentile of Recent Earnings Distribution')
    ax[1, 0].set_ylabel('Kurtosis of 5-year earnings arc-percent change')
    #ax[1, 0].set_ylim(1.5, 6.5)
    ax[1, 0].set_yticks(np.linspace(1.5, 6.5, 11))

    #LE_x_axis = [1, 3.5, 8, 15.5, 25.5, 35.5, 45.5, 55.5, 65.5, 75.5, 85.5, 93, 96.5, 98.5, 100]
    LE_x_axis = [1, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 97, 99, 100]

    ax[1, 1].plot(LE_x_axis, log_growth, color='blue')
    ax[1, 1].set_title('Lifecycle Earnings Growth')
    ax[1, 1].set_xlabel('Percentile of Lifetime Earnings Distribution')
    ax[1, 1].set_ylabel('Log Average Earnings Growth Age 25 to last')
    #ax[1, 1].set_ylim(-1, 3.5)

    ax[2, 0].plot(years_employed, cum_prob, color='blue')
    ax[2, 0].set_title('Lifecycle Nonemployment Distribution')
    ax[2, 0].set_xlabel('Total Years Employed')
    ax[2, 0].set_ylabel('Employment CDF')

    ax[2, 1].plot(range(25, 25+num_years), cohort_variances, color='blue')
    #ax[2, 1].set_ylim(0, 2.2)
    ax[2, 1].set_title('Variance of Log Earnings')
    ax[2, 1].set_xlabel('Age')
    ax[2, 1].set_ylabel('Within-Cohort Variance of Log Earnings')

    if sex == 0:
        ax[0, 0].plot(RE_x_axis, np.array(df_guv_three['std']), color="black", linestyle="--")
        ax[0, 1].plot(RE_x_axis, np.array(df_guv_three['skew']), color="black", linestyle="--", label = "Data")
        ax[1, 0].plot(RE_x_axis, np.array(df_guv_three['kurt']), color="black", linestyle="--")
        ax[1, 1].plot(LE_x_axis, np.array(guv_growth), color="black", linestyle="--")
        ax[2, 0].plot(range(num_years + 1), emp_cdf_truncate, color="black", linestyle="--")
        ax[2, 1].plot(range(25, 25+num_years), var_lny[:(num_years)], color="black", linestyle="--")

    elif sex == 1:
        ax[0, 0].plot(np.array(df_guvf_three['RE']), np.array(df_guvf_three['std']), color="black", linestyle="--")
        ax[0, 1].plot(np.array(df_guvf_three['RE']), np.array(df_guvf_three['skew']), color="black", linestyle="--", label = "Data")
        ax[1, 0].plot(np.array(df_guvf_three['RE']), np.array(df_guvf_three['kurt']), color="black", linestyle="--")
        ax[1, 1].plot(np.array(guvf_growth['LE']), np.array(guvf_growth['LE_growth']), color="black", linestyle="--")

    ax[0, 1].legend()

    plt.suptitle("Our simulated income changes v data: key statistics", fontsize = 20)

    plt.tight_layout(pad = 2)